In [1]:
import pandas as pd

df = pd.read_csv('fraud_detection_transactions.csv')

# Z-score calculate karo transaction_amount ka
mean_amount = df['transaction_amount'].mean()
std_amount = df['transaction_amount'].std()

df['z_score'] = (df['transaction_amount'] - mean_amount) / std_amount

print(df[['transaction_amount', 'z_score', 'is_fraud']].head(10))

   transaction_amount   z_score  is_fraud
0               35.47 -0.547165         0
1               65.02 -0.454012         0
2               69.67 -0.439353         0
3               68.32 -0.443609         0
4             1112.28  2.847363         0
5              155.13 -0.169950         0
6               99.99 -0.343773         0
7               45.35 -0.516019         0
8               41.75 -0.527368         0
9              240.56  0.099359         0


In [2]:
df['flagged_suspicious'] = (df['z_score'] > 2).astype(int)

print("Total flagged as suspicious:", df['flagged_suspicious'].sum())
print("Total actual fraud:", df['is_fraud'].sum())

# Confusion matrix jaisa cross-tab
print(pd.crosstab(df['flagged_suspicious'], df['is_fraud']))

Total flagged as suspicious: 441
Total actual fraud: 170
is_fraud               0   1
flagged_suspicious          
0                   9464  95
1                    366  75


In [3]:
TP = 75
FP = 366
FN = 95

precision = TP / (TP + FP)
recall = TP / (TP + FN)

print("Precision:", precision)
print("Recall:", recall)

Precision: 0.17006802721088435
Recall: 0.4411764705882353


In [4]:
thresholds = [1.5, 2, 2.5, 3, 3.5]

for t in thresholds:
    df['flagged'] = (df['z_score'] > t).astype(int)
    TP = ((df['flagged']==1) & (df['is_fraud']==1)).sum()
    FP = ((df['flagged']==1) & (df['is_fraud']==0)).sum()
    FN = ((df['flagged']==0) & (df['is_fraud']==1)).sum()
    precision = TP / (TP + FP) if (TP+FP) > 0 else 0
    recall = TP / (TP + FN) if (TP+FN) > 0 else 0
    print(f"Threshold {t}: Precision={precision:.3f}, Recall={recall:.3f}, Flagged={df['flagged'].sum()}")

Threshold 1.5: Precision=0.130, Recall=0.500, Flagged=654
Threshold 2: Precision=0.170, Recall=0.441, Flagged=441
Threshold 2.5: Precision=0.215, Recall=0.376, Flagged=297
Threshold 3: Precision=0.290, Recall=0.318, Flagged=186
Threshold 3.5: Precision=0.383, Recall=0.288, Flagged=128


In [5]:
avg_fraud_loss = 1014.40  # cost per missed fraud (false negative)
review_cost = 50          # cost per false alarm (false positive)

total_fraud = df['is_fraud'].sum()

for t in thresholds:
    df['flagged'] = (df['z_score'] > t).astype(int)
    TP = ((df['flagged']==1) & (df['is_fraud']==1)).sum()
    FP = ((df['flagged']==1) & (df['is_fraud']==0)).sum()
    FN = ((df['flagged']==0) & (df['is_fraud']==1)).sum()
    
    total_cost = (FN * avg_fraud_loss) + (FP * review_cost)
    print(f"Threshold {t}: FN={FN}, FP={FP}, Total Cost=₹{total_cost:,.0f}")

Threshold 1.5: FN=85, FP=569, Total Cost=₹114,674
Threshold 2: FN=95, FP=366, Total Cost=₹114,668
Threshold 2.5: FN=106, FP=233, Total Cost=₹119,176
Threshold 3: FN=116, FP=132, Total Cost=₹124,270
Threshold 3.5: FN=121, FP=79, Total Cost=₹126,692
